> Projeto Desenvolve <br>
Programação Intermediária com Python <br>
Profa. Camila Laranjeira (mila@projetodesenvolve.com.br) <br>

# 3.14 - ORM

## Exercícios

#### Q1. Conhecendo os dados
Baixe o seguinte csv onde iremos trabalhar. Ele contém informações sobre salários de profissionais de dados de uma empresa hipotética entre 2009 e 2016
* https://github.com/camilalaranjeira/python-intermediario/blob/main/salaries.csv

Suas colunas, descritas na [página do Kaggle que contém o dataset](https://www.kaggle.com/datasets/krishujeniya/salary-prediction-of-data-professions?resource=download), são:
* FIRST NAME: Primeiro nome do profissional de dados (String)
* LAST NAME: Sobrenome do profissional de dados (String)
* SEX: Gênero do profissional de dados (String: 'F' para Feminino, 'M' para Masculino)
* DOJ (Date of Joining): A data em que o profissional de dados ingressou na empresa (Data no formato MM/DD/AAAA)
* CURRENT DATE: A data atual ou a data de referência dos dados (Data no formato MM/DD/AAAA)
* DESIGNATION: O cargo ou designação do profissional de dados (String: ex., Analista, Analista Sênior, Gerente)
* AGE: Idade do profissional de dados (Integer)
* SALARY: Salário anual do profissional de dados (Float)
* UNIT: Unidade de negócios ou departamento em que o profissional de dados trabalha (String: ex., TI, Finanças, Marketing)
* LEAVES USED: Número de licenças utilizadas pelo profissional de dados (Integer)
* LEAVES REMAINING: Número de licenças restantes para o profissional de dados (Integer)
* RATINGS: Avaliações de desempenho do profissional de dados (Float)
* PAST EXP: Experiência de trabalho anterior em anos antes de ingressar na empresa atual (Float)

Na célula a seguir, **carregue os dados do CSV e dê uma olhada neles antes de seguir**.

In [23]:
### Escreva sua resposta aqui

import pandas as pd

# Lendo o arquivo CSV
df = pd.read_csv('salaries.csv')

# Exibindo as primeiras linhas e informações gerais do DataFrame
display(df.head())
df.info()

,FIRST NAME,LAST NAME,SEX,DOJ,CURRENT DATE,DESIGNATION,AGE,SALARY,UNIT,LEAVES USED,LEAVES REMAINING,RATINGS,PAST EXP
0,TOMASA,ARMEN,F,5-18-2014,01-07-2016,Analyst,21.0,44570,Finance,24.0,6.0,2.0,0
1,ANNIE,NaN,F,NaN,01-07-2016,Associate,NaN,89207,Web,NaN,13.0,NaN,7
2,OLIVE,ANCY,F,7-28-2014,01-07-2016,Analyst,21.0,40955,Finance,23.0,7.0,3.0,0
3,CHERRY,AQUILAR,F,04-03-2013,01-07-2016,Analyst,22.0,45550,IT,22.0,8.0,3.0,0
4,LEON,ABOULAHOUD,M,11-20-2014,01-07-2016,Analyst,NaN,43161,Operations,27.0,3.0,NaN,3


<class 'pandas.DataFrame'>
RangeIndex: 2639 entries, 0 to 2638
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   FIRST NAME        2639 non-null   str    
 1   LAST NAME         2637 non-null   str    
 2   SEX               2639 non-null   str    
 3   DOJ               2638 non-null   str    
 4   CURRENT DATE      2639 non-null   str    
 5   DESIGNATION       2639 non-null   str    
 6   AGE               2636 non-null   float64
 7   SALARY            2639 non-null   int64  
 8   UNIT              2639 non-null   str    
 9   LEAVES USED       2636 non-null   float64
 10  LEAVES REMAINING  2637 non-null   float64
 11  RATINGS           2637 non-null   float64
 12  PAST EXP          2639 non-null   int64  
dtypes: float64(4), int64(2), str(7)
memory usage: 268.2 KB


#### Q2. Modelando os dados
Você deve **criar um ORM com SQLAlchemy capaz de comportar os dados dessa base**.

* Crie um campo de chave primária `ID`, que deve ser incrementado automaticamente
* Os campos SEX, DESIGNATION e UNIT devem ser definidos como classes `Enum` com os possíveis valores (consulte os valores únicos dessas colunas)
* Para os outros campos, consulte os tipos de dados informados na descrição acima

In [24]:
### Escreva sua resposta aqui

import enum
from sqlalchemy.orm import declarative_base
from sqlalchemy import Column, Integer, String, Float, Date, Enum

Base = declarative_base()

# Definindo as classes Enum com base nos valores únicos dos dados
class SexEnum(enum.Enum):
    F = 'F'
    M = 'M'

class DesignationEnum(enum.Enum):
    Analyst = 'Analyst'
    Associate = 'Associate'
    Senior_Analyst = 'Senior Analyst'
    Senior_Manager = 'Senior Manager'
    Manager = 'Manager'
    Director = 'Director'

class UnitEnum(enum.Enum):
    Finance = 'Finance'
    Web = 'Web'
    IT = 'IT'
    Operations = 'Operations'
    Marketing = 'Marketing'
    Management = 'Management'

# Criando o modelo ORM
class SalaryProfile(Base):
    __tablename__ = 'salaries'
    
    id = Column('ID', Integer, primary_key=True, autoincrement=True)
    first_name = Column('FIRST NAME', String)
    last_name = Column('LAST NAME', String)
    
    sex = Column('SEX', Enum(SexEnum, values_callable=lambda obj: [e.value for e in obj]))
    doj = Column('DOJ', Date)
    current_date = Column('CURRENT DATE', Date)
    designation = Column('DESIGNATION', Enum(DesignationEnum, values_callable=lambda obj: [e.value for e in obj]))
    age = Column('AGE', Integer)
    salary = Column('SALARY', Float)
    unit = Column('UNIT', Enum(UnitEnum, values_callable=lambda obj: [e.value for e in obj]))
    
    leaves_used = Column('LEAVES USED', Integer)
    leaves_remaining = Column('LEAVES REMAINING', Integer)
    ratings = Column('RATINGS', Float)
    past_exp = Column('PAST EXP', Float)

#### Q3. Estabelecendo uma conexão

Usando o método `create_engine` do SQLAlchemy, crie uma conexão com um novo banco de dados SQLite chamado `salarios`.

In [25]:
### Escreva sua resposta aqui

from sqlalchemy import create_engine

# Conectando com um banco SQLite chamado "salarios"
engine = create_engine('sqlite:///salarios.db')

#### Q4. Criando as tabelas
Crie as tabelas da questão Q2 no banco `salarios`.

In [26]:
### Escreva sua resposta aqui

# Cria as tabelas do modelo no banco de dados (se não existirem)
Base.metadata.create_all(engine)

#### Q5. Populando

Usando o método `to_sql` da biblioteca Pandas (veja [a documentação](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html)), popule o banco `salarios` com os dados do csv que você carregou na questão Q1.
* Lembre-se de definir o parâmetro `if_exists='append'` para que as tabelas não sejam dropadas e recriadas.

In [27]:
### Escreva sua resposta aqui

# Convertendo as colunas de data (tratando valores nulos com coerce)
df['DOJ'] = pd.to_datetime(df['DOJ'], errors='coerce')
df['CURRENT DATE'] = pd.to_datetime(df['CURRENT DATE'], errors='coerce')

# Populando o banco com a tabela do Pandas
df.to_sql('salaries', con=engine, if_exists='append', index=False)

print("Dados inseridos com sucesso!")

Dados inseridos com sucesso!


#### Q6. Consultas SQL vs ORM

Agrupe os dados por DESIGNATION e selecione o mínimo, máximo e a média dos salários (SALARY) divididos por 12. Já que o atributo SALARY é anual, dividir por 12 nos mostrará os valores mensais.

Assumindo que a variável que armazena a sua conexão se chama `engine`, você deve realizar a query acima de três formas:
* Executando a query SQL através de uma instância de conexão retornada pelo método `engine.connect()`
* Executando a query SQL com o método `read_sql_query` do Pandas (veja [a documentação](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html)). Você usará mesma instância `engine.connect()` como um dos parâmetros.
* Executando uma query criada com o módulo `select` do SQLAlchemy. Sua execução deve ser feita através de um objeto `Session` do módulo `orm` do SQLAlchemy (`Session(engine)`).


In [28]:
### Execute aqui sua query SQL com SQLAlchemy

from sqlalchemy import text

query = text("""
    SELECT DESIGNATION, 
           MIN(SALARY)/12.0 AS min_mensal, 
           MAX(SALARY)/12.0 AS max_mensal, 
           AVG(SALARY)/12.0 AS avg_mensal 
    FROM salaries 
    GROUP BY DESIGNATION
""")

with engine.connect() as conn:
    result = conn.execute(query)
    print("--- Resultado via SQLAlchemy Exec. ---")
    for row in result:
        print(row)

--- Resultado via SQLAlchemy Exec. ---
('Analyst', 3333.4166666666665, 4165.0, 3751.675987685993)
('Associate', 5846.166666666667, 8300.25, 7266.915094339623)
('Director', 17832.25, 32342.666666666668, 23914.265625)
('Manager', 8343.666666666666, 12407.5, 10522.716049382716)
('Senior Analyst', 4170.333333333333, 5830.5, 4991.778792134832)
('Senior Manager', 12614.416666666666, 16631.416666666668, 14888.689516129032)


In [29]:
### Execute aqui sua query SQL com SQLAlchemy + Pandas
with engine.connect() as conn:
    df_resultado = pd.read_sql_query(query, conn)

print("--- Resultado via Pandas read_sql_query ---")
display(df_resultado)

--- Resultado via Pandas read_sql_query ---


,DESIGNATION,min_mensal,max_mensal,avg_mensal
0,Analyst,3333.416667,4165.000000,3751.675988
1,Associate,5846.166667,8300.250000,7266.915094
2,Director,17832.250000,32342.666667,23914.265625
3,Manager,8343.666667,12407.500000,10522.716049
4,Senior Analyst,4170.333333,5830.500000,4991.778792
5,Senior Manager,12614.416667,16631.416667,14888.689516


In [30]:
### Execute aqui sua query com SQLAlchemy ORM

from sqlalchemy.orm import Session
from sqlalchemy import select, func

with Session(engine) as session:
    # Construindo a query com as funções e divisões
    stmt = select(
        SalaryProfile.designation,
        (func.min(SalaryProfile.salary) / 12).label('min_mensal'),
        (func.max(SalaryProfile.salary) / 12).label('max_mensal'),
        (func.avg(SalaryProfile.salary) / 12).label('avg_mensal')
    ).group_by(SalaryProfile.designation)
    
    result = session.execute(stmt)
    
    print("--- Resultado via SQLAlchemy ORM ---")
    for row in result:
        # Acessando pelo nome original da Enum ou valor da String
        print(f"Cargo: {row.designation.value if isinstance(row.designation, enum.Enum) else row.designation} | " 
              f"Min: {row.min_mensal:.2f} | Max: {row.max_mensal:.2f} | Média: {row.avg_mensal:.2f}")


--- Resultado via SQLAlchemy ORM ---
Cargo: Analyst | Min: 3333.42 | Max: 4165.00 | Média: 3751.68
Cargo: Associate | Min: 5846.17 | Max: 8300.25 | Média: 7266.92
Cargo: Director | Min: 17832.25 | Max: 32342.67 | Média: 23914.27
Cargo: Manager | Min: 8343.67 | Max: 12407.50 | Média: 10522.72
Cargo: Senior Analyst | Min: 4170.33 | Max: 5830.50 | Média: 4991.78
Cargo: Senior Manager | Min: 12614.42 | Max: 16631.42 | Média: 14888.69
